In [10]:
import requests
import os
from cairosvg import svg2png
from PIL import Image
import io

# Mapping from COCO category names to FontAwesome icon names (free solid set)
# You'll need to manually curate this - here's a partial example
COCO_TO_FA = {
    'person': 'user',
    'bicycle': 'bicycle',
    'car': 'car',
    'motorcycle': 'motorcycle',
    'airplane': 'plane',
    'bus': 'bus',
    'train': 'train',
    'truck': 'truck',
    'boat': 'ship',
    'bird': 'dove',
    'cat': 'cat',
    'dog': 'dog',
    'horse': 'horse',
    'cow': 'cow',
    'sheep': 'hippo',  # no sheep icon, approximate
    'elephant': 'elephant',  # pro only, actually
    'umbrella': 'umbrella',
    'handbag': 'bag',
    'suitcase': 'suitcase',
    'chair': 'chair',
    'couch': 'couch',
    'bed': 'bed',
    'toilet': 'toilet',
    'laptop': 'laptop',
    'keyboard': 'keyboard',
    'phone': 'phone',
    'tv': 'tv',
    'book': 'book',
    'clock': 'clock',
    'scissors': 'scissors',
        # Exact matches
    'bicycle': 'bicycle',
    'car': 'car',
    'bus': 'bus',
    'cat': 'cat',
    'dog': 'dog',
    'horse': 'horse',
    'cow': 'cow',
    'carrot': 'carrot',
    'hotdog': 'hotdog',
    'cake': 'cake',
    'chair': 'chair',
    'couch': 'couch',
    'bed': 'bed',
    'book': 'book',
    'clock': 'clock',
    

    'stop sign': 'circle-stop',
    'fire hydrant': 'fire-extinguisher',
    'sports ball': 'basketball',
    'baseball bat': 'baseball-bat-ball',
    'baseball glove': 'baseball',
    'bottle': 'bottle-water',
    'wine glass': 'glass-martini',
    'bowl': 'bowl',
    'apple': 'apple',
    'mouse': 'computer-mouse',
    'laptop': 'laptop',  # try without 'house-' prefix first
    'scissors': 'hand-scissors',  # try direct, else 'hand-scissors'
    'cell phone': 'mobile-screen-button',  # or 'mobile'

}

def download_fa_icon(icon_name, output_path, size=64):
    """Download from FontAwesome GitHub repo (free icons)"""
    url = f"https://raw.githubusercontent.com/FortAwesome/Font-Awesome/master/svgs/solid/{icon_name}.svg"
    resp = requests.get(url)
    if resp.status_code == 200:
        png_data = svg2png(bytestring=resp.content, output_width=size, output_height=size)
        img = Image.open(io.BytesIO(png_data))
        img.save(output_path)
        return True
    return False

# Download available icons
os.makedirs('coco_icons', exist_ok=True)
results_list = []
for coco_name, fa_name in COCO_TO_FA.items():
    success = download_fa_icon(fa_name, f'coco_icons/{coco_name}.png')
    results_list.append(f"{coco_name}: {'✓' if success else '✗'}")
results_list   

['person: ✓',
 'bicycle: ✓',
 'car: ✓',
 'motorcycle: ✓',
 'airplane: ✓',
 'bus: ✓',
 'train: ✓',
 'truck: ✓',
 'boat: ✓',
 'bird: ✓',
 'cat: ✓',
 'dog: ✓',
 'horse: ✓',
 'cow: ✗',
 'sheep: ✓',
 'elephant: ✗',
 'umbrella: ✓',
 'handbag: ✗',
 'suitcase: ✓',
 'chair: ✓',
 'couch: ✓',
 'bed: ✓',
 'toilet: ✓',
 'laptop: ✓',
 'keyboard: ✓',
 'phone: ✓',
 'tv: ✓',
 'book: ✓',
 'clock: ✓',
 'scissors: ✓',
 'carrot: ✓',
 'hotdog: ✓',
 'cake: ✗',
 'stop sign: ✗',
 'fire hydrant: ✓',
 'sports ball: ✗',
 'baseball bat: ✗',
 'baseball glove: ✗',
 'bottle: ✗',
 'wine glass: ✓',
 'bowl: ✗',
 'apple: ✗',
 'mouse: ✗',
 'cell phone: ✗']

In [8]:
import requests

# Get full list of available icons
url = "https://api.github.com/repos/FortAwesome/Font-Awesome/contents/svgs/solid"
resp = requests.get(url)
available = [f['name'].replace('.svg', '') for f in resp.json()]

# All 80 COCO categories
coco_categories = [
    'person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus', 'train', 'truck', 'boat',
    'traffic light', 'fire hydrant', 'stop sign', 'parking meter', 'bench', 'bird', 'cat',
    'dog', 'horse', 'sheep', 'cow', 'elephant', 'bear', 'zebra', 'giraffe', 'backpack',
    'umbrella', 'handbag', 'tie', 'suitcase', 'frisbee', 'skis', 'snowboard', 'sports ball',
    'kite', 'baseball bat', 'baseball glove', 'skateboard', 'surfboard', 'tennis racket',
    'bottle', 'wine glass', 'cup', 'fork', 'knife', 'spoon', 'bowl', 'banana', 'apple',
    'sandwich', 'orange', 'broccoli', 'carrot', 'hot dog', 'pizza', 'donut', 'cake',
    'chair', 'couch', 'potted plant', 'bed', 'dining table', 'toilet', 'tv', 'laptop',
    'mouse', 'remote', 'keyboard', 'cell phone', 'microwave', 'oven', 'toaster', 'sink',
    'refrigerator', 'book', 'clock', 'vase', 'scissors', 'teddy bear', 'hair drier', 'toothbrush'
]

def find_icon(category, available):
    """Try to find a matching icon"""
    clean = category.replace(' ', '-')
    if clean in available:
        return ('exact', clean)
    
    nospace = category.replace(' ', '')
    if nospace in available:
        return ('exact', nospace)
    
    words = category.split()
    for word in words:
        if len(word) >= 3:
            matches = [a for a in available if word in a]
            if matches:
                return ('candidates', matches[:8])
    
    return ('none', None)

# Write results to file
with open('icon_search_results.txt', 'w') as f:
    for cat in coco_categories:
        result_type, result = find_icon(cat, available)
        if result_type == 'exact':
            f.write(f"{cat}: EXACT -> {result}\n")
        elif result_type == 'candidates':
            f.write(f"{cat}: CANDIDATES -> {result}\n")
        else:
            f.write(f"{cat}: NO MATCH\n")

print("Results written to icon_search_results.txt")

Results written to icon_search_results.txt
